In [ ]:
"""
Experiment Runner for VLM Model Evaluation on Cauldron Dataset.

This module provides comprehensive data collection for VLM routing research:
- Runs inference across multiple VLM models
- Captures input features, output data, quality metrics, cost metrics
- Supports all 50 Cauldron configs
- Saves results in parquet format for analysis

Usage:
    from experiment_runner import ExperimentRunner
    
    runner = ExperimentRunner.from_yaml("configs/inference_vlm.yaml")
    results = runner.run_experiment(
        configs=["textvqa", "chartqa", "ai2d"],
        samples_per_config=100
    )
"""


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:

from __future__ import annotations

import json

import time
import uuid
from pathlib import Path
from PIL import Image
from datetime import datetime

from typing import Any, Dict, List, Optional, Tuple, Union
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

from datasets import load_dataset, get_dataset_config_names
from tqdm.auto import tqdm

import os, sys
# Add parent to path for local imports
sys.path.append(os.path.abspath(".."))
# Local imports
from inference_api_call.client import WhichVLMClient


In [ ]:
from imports.config import ALL_CAULDRON_CONFIGS
from imports.config import ExperimentConfig, SampleRecord

In [ ]:
from imports.modules import FeatureExtractor, compute_routing_labels, analyze_model_strengths
from imports.evaluation import SemanticF1Evaluator, Scorer, GliderEvaluator
from imports.dataset_loader import CauldronLoader

In [ ]:
from openai import OpenAI

# Create a client pointing to your vLLM server
client = OpenAI(
    base_url="http://localhost:8805/v1",
    api_key="dummy"  # vLLM does not check keys
)
MODEL_NAME="PatronusAI/glider"
def chat_fn(messages, max_tokens=512, model_name=MODEL_NAME):
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.0,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content
# semantic_f1_evaluator = SemanticF1Evaluator(chat_fn) 
glider_evaluator = GliderEvaluator(chat_fn)

In [ ]:

# =============================================================================
# EXPERIMENT RUNNER
# =============================================================================

class ExperimentRunner:
    """
    Main experiment runner for VLM evaluation.
    
    Handles:
    - Loading dataset samples
    - Running inference across all VLM models
    - Capturing comprehensive metrics
    - Saving results to parquet
    """
    
    def __init__(
        self,
        client: WhichVLMClient,
        config: Optional[ExperimentConfig] = None,
    ):
        self.client = client
        self.config = config or ExperimentConfig()
        self.feature_extractor = FeatureExtractor()
        self.scorer = Scorer()
        # self.semantic_f1_evaluator = semantic_f1_evaluator
        self.glider_evaluator = glider_evaluator
        
        # Create output directories
        self.output_dir = Path(self.config.output_dir)
        self.images_dir = self.output_dir / "images"
        self.results_dir = self.output_dir / "runs" / self.config.run_id
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.images_dir.mkdir(parents=True, exist_ok=True)
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        # Save config
        config_path = self.results_dir / "config.json"
        with open(config_path, 'w') as f:
            json.dump(self.config.to_dict(), f, indent=2)
    
    @classmethod
    def from_yaml(
        cls,
        yaml_path: str,
        config: Optional[ExperimentConfig] = None,
    ) -> "ExperimentRunner":
        """Create runner from a YAML config file."""
        client = WhichVLMClient.from_yaml(yaml_path)
        return cls(client=client, config=config)
    
    def _save_image(self, image: Image.Image, config_name: str, sample_id: str) -> str:
        """Save image to disk and return path."""
        img_dir = self.images_dir / config_name
        img_dir.mkdir(parents=True, exist_ok=True)
        img_path = img_dir / f"{sample_id}.png"
        image.save(img_path)
        return str(img_path)
    
    def _run_single_model(
        self,
        model_name: str,
        qa: Dict[str, Any],
        sample_id: str,
        source_index: int,
        config_name: str,
        image_path: Optional[str],
        image_hash: Optional[str],
        img_features: Dict[str, Any],
        txt_features: Dict[str, Any],
    ) -> SampleRecord:
        """Run inference for a single model on a single sample."""
        
        start_time = time.perf_counter()
        
        try:
            result = self.client.runner.chat(
                model_name=model_name,
                messages=self.client.runner.run_all(
                    prompt=qa['prompt'],
                    images=qa['image'],
                    model_names=[model_name]
                ).get(model_name, {}).get('request', {}).get('messages', []),
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
                top_p=self.config.top_p,
            )
        except Exception as e:
            # If direct chat fails, use the VLM suite
            results = self.client.vlm.run_image(
                image=qa['image'],
                text=qa['prompt'],
                models=[model_name],
                temperature=self.config.temperature,
                max_tokens=self.config.max_tokens,
                top_p=self.config.top_p,
            )
            result = results.get(model_name, {'ok': False, 'error': str(e)})
        
        latency_ms = (time.perf_counter() - start_time) * 1000
        
        # Extract response data
        ok = result.get('ok', False)
        response_raw = result.get('response_text', '') if ok else None
        error_message = result.get('error') if not ok else None
        
        # Get usage info
        usage = result.get('usage', {})
        input_tokens = usage.get('prompt_tokens')
        output_tokens = usage.get('completion_tokens')
        total_tokens = usage.get('total_tokens')
        
        # Compute scores
        scores = self.scorer.compute_all_scores(
            pred=response_raw or '',
            gt=qa['ground_truth'],
            gt_type=qa['ground_truth_type']
        )
        
        # Get model info
        model_endpoint = self.client.runner.models.get(model_name)
        model_id = model_endpoint.model_id if model_endpoint else model_name
        
        # Estimate cost (all zeros for local models)
        est_cost = result.get('est_cost', 0.0)
        
        return SampleRecord(
            # Identity
            sample_id=sample_id,
            run_id=self.config.run_id,
            timestamp_utc=datetime.utcnow().isoformat(),
            
            # Input data
            image_path=image_path,
            image_bytes_hash=image_hash,
            prompt_raw=qa['prompt'],
            prompt_formatted=None,  # Could add model-specific formatting
            system_prompt=None,
            source_dataset=f"cauldron_{config_name}",
            source_config=config_name,
            router_task=qa['router_task'],
            ground_truth=qa['ground_truth'],
            ground_truth_type=qa['ground_truth_type'],
            mc_options=qa.get('mc_options'),
            source_index=source_index,
            
            # Input features
            img_width=img_features.get('img_width'),
            img_height=img_features.get('img_height'),
            img_aspect_ratio=img_features.get('img_aspect_ratio'),
            img_file_size_bytes=img_features.get('img_file_size_bytes'),
            txt_prompt_length_chars=txt_features['txt_prompt_length_chars'],
            txt_prompt_length_words=txt_features['txt_prompt_length_words'],
            txt_question_type=txt_features['txt_question_type'],
            txt_has_mc_options=txt_features['txt_has_mc_options'],
            
            # Model info
            model_name=model_name,
            model_id=model_id,
            
            # Output data
            response_raw=response_raw,
            response_parsed=response_raw,  # Could add parsing logic
            response_length_chars=len(response_raw) if response_raw else 0,
            response_length_tokens=output_tokens,
            stop_reason=None,
            error_message=error_message,
            is_refusal=scores['is_refusal'],
            ok=ok,
            
            # Quality scores
            score_exact_match=scores['score_exact_match'],
            score_exact_match_normalized=scores['score_exact_match_normalized'],
            score_contains_gt=scores['score_contains_gt'],
            score_gt_in_response=scores['score_gt_in_response'],
            score_f1=scores['score_f1'],
            score_numeric_match=scores['score_numeric_match'],
            score_mc_letter_match=scores['score_mc_letter_match'],
            is_correct=scores['is_correct'],
            pred_answer_letter=scores['pred_answer_letter'],
            gt_answer_letter=scores['gt_answer_letter'],
            
            # Cost metrics
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            total_tokens=total_tokens,
            latency_ms=latency_ms,
            estimated_cost_usd=est_cost,
            
            # Inference config
            inference_temperature=self.config.temperature,
            inference_max_tokens=self.config.max_tokens,
            inference_top_p=self.config.top_p,
        )
    
    def evaluate_sample(
        self,
        qa: Dict[str, Any],
        sample_id: str,
        source_index: int,
        config_name: str,
    ) -> List[SampleRecord]:
        """Evaluate a single sample across all VLM models."""
        
        # Save image if configured
        image_path = None
        if self.config.save_images:
            image_path = self._save_image(qa['image'], config_name, sample_id)
        
        # Extract features
        image_hash = self.feature_extractor.compute_image_hash(qa['image'])
        img_features = self.feature_extractor.extract_image_features(qa['image'], image_path)
        txt_features = self.feature_extractor.extract_text_features(qa['prompt'])
        
        # Run all VLM models
        records = []
        vlm_models = self.client.list_vlm_models()
        
        for model_name in vlm_models:
            try:
                # Use VLM suite directly
                results = self.client.vlm.run_image(
                    image=qa['image'],
                    text=qa['prompt'],
                    models=[model_name],
                    temperature=self.config.temperature,
                    max_tokens=self.config.max_tokens,
                    top_p=self.config.top_p,
                )
                result = results.get(model_name, {'ok': False, 'error': 'No result'})
                
                latency_ms = result.get('latency_ms', 0)
                ok = result.get('ok', False)
                response_raw = result.get('response_text', '') if ok else None
                error_message = result.get('error') if not ok else None
                
                usage = result.get('usage', {})
                input_tokens = usage.get('prompt_tokens')
                output_tokens = usage.get('completion_tokens')
                total_tokens = usage.get('total_tokens')
                est_cost = result.get('est_cost', 0.0)
                
            except Exception as e:
                latency_ms = 0
                ok = False
                response_raw = None
                error_message = str(e)
                input_tokens = None
                output_tokens = None
                total_tokens = None
                est_cost = 0.0
            
            print(f"Evaluating model: {model_name} on sample: {sample_id}")
            print("Prompt:", qa['prompt'])
            print("Current Query Response:", response_raw)
            # semantic_results = self.semantic_f1_evaluator.evaluate_answer(
            #         question=qa['prompt'],
            #         answer=response_raw or '',
            #         references=qa['ground_truth']
            #         )
            eval_res = glider_evaluator.evaluate(
                    question=qa['prompt'],
                    model_answer=response_raw or '',
                    ground_truth=qa['ground_truth'],
                    sample_id=sample_id,
                    extra_context=None,
                )
            # print(f"Semantic F1 results: {semantic_results}")
            # Compute scores
            scores = self.scorer.compute_all_scores(
                pred=response_raw or '',
                gt=qa['ground_truth'],
                gt_type=qa['ground_truth_type']
            )
            
            # Get model info
            model_endpoint = self.client.runner.models.get(model_name)
            model_id = model_endpoint.model_id if model_endpoint else model_name
            
            record = SampleRecord(
                # Identity
                sample_id=sample_id,
                run_id=self.config.run_id,
                timestamp_utc=datetime.utcnow().isoformat(),
                
                # Input data
                image_path=image_path,
                image_bytes_hash=image_hash,
                prompt_raw=qa['prompt'],
                prompt_formatted=None,
                system_prompt=None,
                source_dataset=f"cauldron_{config_name}",
                source_config=config_name,
                router_task=qa['router_task'],
                ground_truth=qa['ground_truth'],
                ground_truth_type=qa['ground_truth_type'],
                mc_options=qa.get('mc_options'),
                source_index=source_index,
                
                # Input features
                img_width=img_features.get('img_width'),
                img_height=img_features.get('img_height'),
                img_aspect_ratio=img_features.get('img_aspect_ratio'),
                img_file_size_bytes=img_features.get('img_file_size_bytes'),
                txt_prompt_length_chars=txt_features['txt_prompt_length_chars'],
                txt_prompt_length_words=txt_features['txt_prompt_length_words'],
                txt_question_type=txt_features['txt_question_type'],
                txt_has_mc_options=txt_features['txt_has_mc_options'],
                
                # Model info
                model_name=model_name,
                model_id=model_id,
                
                # Output data
                response_raw=response_raw,
                response_parsed=response_raw,
                response_length_chars=len(response_raw) if response_raw else 0,
                response_length_tokens=output_tokens,
                stop_reason=None,
                error_message=error_message,
                is_refusal=scores['is_refusal'],
                ok=ok,
                
                # Quality scores
                score_exact_match=scores['score_exact_match'],
                score_exact_match_normalized=scores['score_exact_match_normalized'],
                score_contains_gt=scores['score_contains_gt'],
                score_gt_in_response=scores['score_gt_in_response'],
                score_f1=scores['score_f1'],
                score_numeric_match=scores['score_numeric_match'],
                score_mc_letter_match=scores['score_mc_letter_match'],
                is_correct=scores['is_correct'],
                pred_answer_letter=scores['pred_answer_letter'],
                gt_answer_letter=scores['gt_answer_letter'],
                
                # Cost metrics
                input_tokens=input_tokens,
                output_tokens=output_tokens,
                total_tokens=total_tokens,
                latency_ms=latency_ms,
                estimated_cost_usd=est_cost,
                
                # Inference config
                inference_temperature=self.config.temperature,
                inference_max_tokens=self.config.max_tokens,
                inference_top_p=self.config.top_p,
                
                # Semantic F1 results
                # semantic_f1_precision=semantic_results.get('precision'),
                # semantic_f1_recall=semantic_results.get('recall'),
                # semantic_f1_f1=semantic_results.get('f1'),
                # semantic_f1_gen_statements=semantic_results.get('generated_statements'),
                # semantic_f1_gt_statements=semantic_results.get('ground_truth_statements'),
                # semantic_f1_matches=semantic_results.get('matches'),
                # semantic_f1_labels=semantic_results.get('labels'),
                
                # Glider evaluation results
                glider_score=eval_res.get("score"),
                glider_reasoning=eval_res.get("reasoning"),
                glider_highlight=eval_res.get("highlight"),
                glider_raw_output=eval_res.get("raw_output"),
            )
            records.append(record)
        
        return records
    
    def evaluate_config(
        self,
        config_name: str,
        n_samples: int = 100,
        progress_bar: bool = True,
    ) -> pd.DataFrame:
        """Evaluate all VLM models on samples from a single Cauldron config."""
        
        print(f"\n{'='*60}")
        print(f"Evaluating: {config_name}")
        print(f"{'='*60}")
        
        # Load samples
        samples = CauldronLoader.load_samples(config_name, n_samples)
        print(f"Loaded {len(samples)} samples")
        
        all_records = []
        iterator = tqdm(enumerate(samples), total=len(samples), desc=config_name) if progress_bar else enumerate(samples)
        
        for idx, sample in iterator:
            qa = CauldronLoader.extract_qa(sample, config_name)
            if qa is None:
                continue
            
            sample_id = f"{config_name}_{idx:05d}_{uuid.uuid4().hex[:8]}"
            
            try:
                records = self.evaluate_sample(qa, sample_id, idx, config_name)
                all_records.extend(records)
            except Exception as e:
                print(f"Error on sample {idx}: {e}")
                continue
        
        # Convert to DataFrame
        df = pd.DataFrame([r.to_dict() for r in all_records])
        
        # Save intermediate results
        output_path = self.results_dir / f"{config_name}.parquet"
        df.to_parquet(output_path, index=False)
        print(f"Saved {len(df)} records to {output_path}")
        
        return df
    
    def run_experiment(
        self,
        configs: Optional[List[str]] = None,
        samples_per_config: int = 100,
        progress_bar: bool = True,
    ) -> pd.DataFrame:
        """
        Run full experiment across multiple Cauldron configs.
        
        Args:
            configs: List of config names. If None, uses all configs.
            samples_per_config: Number of samples to evaluate per config.
            progress_bar: Show progress bar.
            
        Returns:
            DataFrame with all results merged.
        """
        if configs is None:
            configs = ALL_CAULDRON_CONFIGS
        
        print(f"\n{'#'*60}")
        print(f"STARTING EXPERIMENT: {self.config.run_id}")
        print(f"{'#'*60}")
        print(f"Configs: {len(configs)}")
        print(f"Samples per config: {samples_per_config}")
        print(f"VLM models: {self.client.list_vlm_models()}")
        print(f"Output dir: {self.results_dir}")
        
        all_dfs = []
        
        for config_name in configs:
            try:
                df = self.evaluate_config(
                    config_name=config_name,
                    n_samples=samples_per_config,
                    progress_bar=progress_bar,
                )
                all_dfs.append(df)
                
                # Print quick summary
                if not df.empty:
                    for model in df['model_name'].unique():
                        model_df = df[df['model_name'] == model]
                        acc = model_df['is_correct'].mean()
                        lat = model_df['latency_ms'].mean()
                        print(f"  {model}: {acc*100:.1f}% accuracy, {lat:.0f}ms avg latency")
                        
            except Exception as e:
                print(f"ERROR on {config_name}: {e}")
                continue
        
        # Merge all results
        if not all_dfs:
            print("No results collected!")
            return pd.DataFrame()
        
        df_all = pd.concat(all_dfs, ignore_index=True)
        
        # Save merged results
        merged_path = self.results_dir / "all_results.parquet"
        df_all.to_parquet(merged_path, index=False)
        print(f"\n{'='*60}")
        print(f"EXPERIMENT COMPLETE")
        print(f"Total records: {len(df_all)}")
        print(f"Saved to: {merged_path}")
        
        # Print final summary
        self._print_summary(df_all)
        
        return df_all
    
    def _print_summary(self, df: pd.DataFrame):
        """Print experiment summary statistics."""
        print(f"\n{'='*60}")
        print("SUMMARY")
        print(f"{'='*60}")
        
        # Overall stats
        print(f"\nTotal samples: {df['sample_id'].nunique()}")
        print(f"Total records: {len(df)}")
        print(f"Configs evaluated: {df['source_config'].nunique()}")
        print(f"Models evaluated: {df['model_name'].nunique()}")
        
        # Per-model stats
        print(f"\n--- Model Performance ---")
        model_stats = df.groupby('model_name').agg({
            'is_correct': 'mean',
            'latency_ms': 'mean',
            'ok': 'mean',
            'sample_id': 'count',
        }).round(4)
        model_stats.columns = ['accuracy', 'avg_latency_ms', 'success_rate', 'n_samples']
        model_stats = model_stats.sort_values('accuracy', ascending=False)
        print(model_stats.to_string())
        
        # Per-task stats
        print(f"\n--- Task Performance (averaged across models) ---")
        task_stats = df.groupby('router_task').agg({
            'is_correct': 'mean',
            'sample_id': 'count',
        }).round(4)
        task_stats.columns = ['avg_accuracy', 'n_records']
        task_stats = task_stats.sort_values('avg_accuracy', ascending=False)
        print(task_stats.head(15).to_string())





In [ ]:
from pathlib import Path
    
# Configuration
config = ExperimentConfig(
    temperature=0.0,
    max_tokens=1024,
    save_images=False,
    output_dir=Path("./experiment_data_final_1"),
)


In [ ]:
# Create runner
CONFIG_PATH = (Path.cwd().parent / "configs" / "inference_vlm.yaml").resolve()
runner = ExperimentRunner.from_yaml(
    yaml_path=str(CONFIG_PATH),
    config=config,
)

In [ ]:
# Run on a subset of configs for testing
# test_configs = ["textvqa", 
#                 "chartqa", 
#                 "ai2d"
#                 ]
df = runner.run_experiment(
    configs=ALL_CAULDRON_CONFIGS,
    samples_per_config=1000,
)

In [ ]:
# Compute routing labels
routing_df = compute_routing_labels(df)
print("\nRouting Labels Sample:")
print(routing_df.head(10))

In [ ]:
# Analyze model strengths
strengths_df = analyze_model_strengths(df)
print("\nModel Strengths by Task:")
print(strengths_df)